# Project Build Commands
Compile the source code

In [27]:
import os
from pathlib import Path
import shutil
import subprocess
# os.chdir('hello-world')
Path.cwd()

PosixPath('/home/fuzzy/projects/C++ 2026/hello-world')

In [28]:
import home
from tools import *

In [29]:
from tools.glob4meson.glob4meson import glob_srcs

In [30]:
START_TAG = "# --- PEROXIDE SOURCE FILES START ---"
END_TAG = "# --- PEROXIDE SOURCE FILES END ---"
SRC_DIR = Path('peroxide')
VAR_NAME = 'peroxide_src_files'
glob_srcs(SRC_DIR, VAR_NAME, START_TAG, END_TAG)

Found 9 source files:
  peroxide/clip.cpp
  peroxide/globals.cpp
  peroxide/keyboard.cpp
  peroxide/main.cpp
  peroxide/midi.cpp
  peroxide/pattern.cpp
  peroxide/peroxide.cpp
  peroxide/player.cpp
  peroxide/song.cpp
meson.build updated with 9 source files.


In [25]:
if build("peroxide", "Wait for a clip name in the queue."): print(f"{CRITICAL_PICT}ERROR!")

✅ File `hw7/hw7.hpp` generated.
✅ Build successful
✅ Jupyter output cleared.
✅ Docs generated
✅ Modified files staged for commit.
✅ Changes commited to repository.
✅ Repository pushed to GitHub.


In [85]:
combine_headers()

✅ File `hw7/hw7.hpp` generated.


0

In [84]:
if compile("hello"): print(f"{CRITICAL_PICT}ERROR!")

✅ Build successful


In [33]:
if compile("peroxide"): print(f"{CRITICAL_PICT}ERROR!")

✅ Build successful


## Meson
Easiest method. Ensures that object files are only rebuilt if necessary.

### Setup

In [6]:
%%bash
# Start from scratch:
rm -rf build
meson setup build

The Meson build system
Version: 1.3.2
Source dir: /home/fuzzy/projects/C++ 2026/hello-world
Build dir: /home/fuzzy/projects/C++ 2026/hello-world/build
Build type: native build
Project name: hello7
Project version: 1.0
C++ compiler for the host machine: c++ (gcc 13.3.0 "c++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0")
C++ linker for the host machine: c++ ld.bfd 2.42
Host machine cpu family: x86_64
Host machine cpu: x86_64
Found pkg-config: YES (/usr/bin/pkg-config) 1.8.1
Run-time dependency gtkmm-4.0 found: YES 4.10.0
Run-time dependency fmt found: YES 9.1.0
Run-time dependency spdlog found: YES 1.12.0
Run-time dependency libmagic found: YES 5.45
Build targets in project: 4

Found ninja-1.11.1 at /usr/bin/ninja


In [4]:
# Reconfigure existing build setup
meson setup --reconfigure . build || true # Reconfigure after changing `meson.build`.


ERROR: Neither source directory '.' nor build directory 'build' contain a build file meson.build.


### Build Executable

In [ ]:
%%bash
meson compile -C build peroxide || true # disposes of `CalledProcessError`

### Build Everything

In [ ]:
def build_all(target, message):
    """ Build the target, test it if it's executable, build
        the docs and publish to GitHub.
    """
    ERROR_CODES = [
        "Success"
        "Out of sync with GitHub"
        "Error staging files for commit"
        "Error commiting changes"
        "Error pushing repository"
        "Unknown"
    ]
    # Command line for meson
    tokens = f"meson compile -C build {TARGET}".split()
    # Run meson
    process = subprocess.run(tokens, **OPTIONS)

    if process.returncode:
        outfile = Path(f"logs/{TARGET}_build.log") # Receives stdout from meson.
        outfile.write_text(process.stdout)
        print(f"{ERROR_PICT}Build error!")
        OUTPUT = read_lines(outfile)
        # This seems to work for meson output. It may work for g++ too.
        print(NEWLINE.join([s for s in OUTPUT[3:] if not (s.startswith('c') or s.startswith('INFO')) and s.find(PARENT)]))
        return process.returncode
    else:
        print(f"{CHECK_PICT}Build successful")
    
    # Testing
    # For now, a successful build means the module passed.
    # A target file with no suffix should be an executable.
    # For this project, for now, executables with 0 arguments, options or input should have 0 output.
    EXE = Path(f"build/{TARGET}")
    if EXE.exists():
        print(f"{INFO_PICT}Testing {TARGET}...")
        process = subprocess.run([f"build/{TARGET}"], **OPTIONS)
        ERROR_CODE = process.returncode
        if ERROR_CODE:
            outfile = Path(f"logs/{TARGET}_test.log")
            print(f"{ERROR_PICT}{TARGET} returned error code: {ERROR_CODE}: {ERROR_CODES[min(ERROR_CODE, len(ERROR_CODES) - 1)]}")
            return ERROR_CODE
        if process.stdout:
            print(f"""stdout:{NEWLINE}{process.stdout}{NEWLINE}""")            
        if process.stderr:
            print(f"""stderr:{NEWLINE}{process.stderr}{NEWLINE}""")
        print(f"{CHECK_PICT}Testing complete.")
    
    # rm -rf docs
    # doxygen
    DOCS = Path("docs")
    if DOCS.exists():
        shutil.rmtree(DOCS, onexc=lambda s: print(f"""{ERROR_PICT}Folder {s} does not exist! 🤨
    """))
    process = subprocess.run(["doxygen"], **OPTIONS)
    if not process.returncode:
        print(f"{CHECK_PICT}Docs generated")
    
    # git
    tokens = ["git", "status"]
    process = subprocess.run(tokens, **OPTIONS)
    ERROR_CODE = process.returncode
    if ERROR_CODE:
        outfile = Path(f"logs/{TARGET}_test.log")
        print(f"{ERROR_PICT}{TARGET} returned error code: {ERROR_CODE}: {ERROR_CODES[min(ERROR_CODE, len(ERROR_CODES) - 1)]}")
    if process.stdout: # This is really long and boring. 🙄
        # print(f"""stdout:{NEWLINE}{process.stdout}{NEWLINE}""")
        outfile = Path(f"logs/{TARGET}_git_output.txt")
        outfile.write_text(process.stdout)
        lines = read_lines(outfile)
        # print(lines[5])
        if not lines[1].startswith("Your branch is up to date with"):
            print(f"{STOP_PICT}Branch is not up to date!")
            return 1
        
    if process.stderr: # Should probably print this no matter what if it exists.
        print(f"""stderr:{NEWLINE}{process.stderr}{NEWLINE}""")

    tokens = ["git", "add", "."]
    process = subprocess.run(tokens, **OPTIONS)
    if process.returncode:
        print(f"{STOP_PICT}Error adding files!")
        return 2
    else: print(f"{CHECK_PICT}Modified files staged for commit.")

    tokens = ["git", "commit", "-m", message]
    process = subprocess.run(tokens, **OPTIONS)
    if process.returncode:
        print(f"{STOP_PICT}Error committing changes!")
        return 3
    else: print(f"{CHECK_PICT}Changes commited to repository.")
        
    tokens = ["git", "push"]
    process = subprocess.run(tokens, **OPTIONS)
    if process.returncode:
        print(f"{STOP_PICT}Error pushing repository to GitHub!")
        return 4
    else: print(f"{CHECK_PICT}Repository pushed to GitHub.")

In [ ]:
# Meson
TARGET = "peroxide" # Change to the desired target from meson.build.

OPTIONS = { # kwargs for `run`
    "text" : True, # Ensures utf-8 encoding.
    "capture_output" : True,
    # "check" : True
}

MESSAGE = "Clear all Jupyter output before pushing!"

build_all(TARGET, MESSAGE)

### Build the project

Also runs `doxygen`. See the [Documentation](docs.ipynb) notebook.

In [ ]:
%%bash
meson compile -C build || true
rm -rf docs
doxygen > logs/doxygen/stdout.log 2> logs/doxygen/stderr.log
echo project\ build\ complete

### Clean the `build` directory

In [ ]:
%%bash
cd build
rm -r *
cd ../

### Build setup

In [ ]:
%%bash
meson setup build

## G++

### Precompiled Header

In [ ]:
%%bash
g++ -std=c++23 \
    -Iinclude -Icontrib \
    -x c++-header include/hw7.hpp \
    -o include/hw7.hpp.gch

### Binary Executable

In [ ]:
%%bash
g++ -std=c++23 source/*.cpp \
    -Wall -Wextra -Wpedantic \
    -Iinclude -Icontrib \
    -o ./build/hello \
    -lspdlog \
    -lfmt 

### Object File

In [71]:
%%bash
g++ -std=c++23 \
    -Wall -Wextra -Wpedantic \
    -c peroxide/peroxide.cpp \
    -Ihw7 -Iperoxide -Icontrib -I/usr/include/ \
    -o ./obj/peroxide.o \
    $(pkg-config --cflags glib-2.0 gtkmm-4.0 fmt spdlog rtmidi) \
|| true

g++: warning: build/libhw7.a: linker input file unused because linking not done


### Debugging Information

In [92]:
%%bash
g++ -g -std=c++23 peroxide/*.cpp \
    -Wall -Wextra -Wpedantic \
    -Iinclude -Icontrib -Ihw7 -Iperoxide \
    $(pkg-config --cflags glib-2.0 gtkmm-4.0 fmt libmagic spdlog) \
    build/libhw7.a \
    $(pkg-config --libs glib-2.0 gtkmm-4.0 fmt libmagic spdlog) \
    -lrtmidi \
    -o ./build/peroxide \
|| true

In [75]:
%%bash
g++ -g -std=c++23 peroxide/*.cpp \
    -Wall -Wextra -Wpedantic \
    -Iinclude -Icontrib -Ihw7 -Iperoxide -I/usr/include/ \
    -o ./build/peroxide \
    $(pkg-config --cflags glib-2.0 gtkmm-4.0 fmt libmagic spdlog) \
    -lrtmidi \
    build/libhw7.a \
|| true

/usr/bin/ld: build/libhw7.a.p/source_fs.cpp.o: in function `magic_type(std::filesystem::__cxx11::path const&)':
/home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:39:(.text+0x36a): undefined reference to `magic_open'
/usr/bin/ld: /home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:41:(.text+0x3d5): undefined reference to `magic_load'
/usr/bin/ld: /home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:43:(.text+0x3f8): undefined reference to `magic_error'
/usr/bin/ld: /home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:44:(.text+0x41b): undefined reference to `magic_close'
/usr/bin/ld: /home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:47:(.text+0x485): undefined reference to `magic_file'
/usr/bin/ld: /home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:50:(.text+0x4aa): undefined reference to `magic_error'
/usr/bin/ld: /home/fuzzy/projects/C++ 2026/hello-world/build/../source/fs.cpp:51:(.text+0x4cd): undefined 

In [ ]:
%%bash
g++ -g ...sources... build/libhw7.a -lrtmidi